In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import os
import numpy as np
import pandas as pd
import glob
from PIL import Image

# --- Cấu hình đường dẫn ---
BASE_PATH = "C:/DUT/Ki 2/Edge AI/dataset/"
TRAIN_DIR = os.path.join(BASE_PATH, "train")
IMG_SIZE = (32, 32)
BATCH_SIZE = 32

# --- Custom Callback để theo dõi kỷ lục Accuracy ---
class AccuracyTracker(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.best_val_acc = 0.0
        self.best_val_loss = 10.0

    def on_epoch_end(self, epoch, logs=None):
        current_val_acc = logs.get("val_accuracy")
        current_val_loss = logs.get("val_loss")
        if current_val_acc > self.best_val_acc:
            diff = current_val_acc - self.best_val_acc
            print(f"\n🔥 Epoch {epoch+1}: Val-Accuracy tăng thêm {diff:.4f}! (Từ {self.best_val_acc:.4f} -> {current_val_acc:.4f})")
            self.best_val_acc = current_val_acc
        else:
            print(f"\n⚠️ Epoch {epoch+1}: Val-Accuracy không tăng (Hiện tại: {current_val_acc:.4f} - Kỷ lục: {self.best_val_acc:.4f})")
        if current_val_loss < self.best_val_loss:
            diff_loss = self.best_val_loss - current_val_loss
            print(f"🔥 Epoch {epoch+1}: Val-Loss giảm thêm {diff_loss:.4f}! (Từ {self.best_val_loss:.4f} -> {current_val_loss:.4f})")
            self.best_val_loss = current_val_loss
        else:
            print(f"⚠️ Epoch {epoch+1}: Val-Loss không giảm (Hiện tại: {current_val_loss:.4f} - Kỷ lục: {self.best_val_loss:.4f})")

# --- Chuẩn bị Data với Augmentation mạnh ---
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y)).cache().shuffle(1000).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y)).cache().prefetch(tf.data.AUTOTUNE)

# --- Kiến trúc tối ưu Accuracy (< 200k params) ---
model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    
    layers.SeparableConv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    
    layers.SeparableConv2D(128, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# --- Huấn luyện ---
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=[
        AccuracyTracker(),
        callbacks.ModelCheckpoint('best_model.h5', monitor='val_accuracy', save_best_only=True, verbose=0),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)
    ],
    verbose=1
)

Found 2000 files belonging to 10 classes.
Using 1600 files for training.
Found 2000 files belonging to 10 classes.
Using 400 files for validation.
Epoch 1/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - accuracy: 0.1903 - loss: 2.2213


🔥 [KỶ LỤC MỚI] Epoch 1: Val-Accuracy tăng thêm 0.1175! (Từ 0.0000 -> 0.1175)
50/50 ━━━━━━━━━━━━━━━━━━━━ 8s 125ms/step - accuracy: 0.1916 - loss: 2.2175 - val_accuracy: 0.1175 - val_loss: 2.3041 - learning_rate: 0.0010
Epoch 2/50
49/50 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.3455 - loss: 1.6902
⚠️ Epoch 2: Val-Accuracy không tăng (Hiện tại: 0.1050 - Kỷ lục: 0.1175)
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - accuracy: 0.3460 - loss: 1.6885 - val_accuracy: 0.1050 - val_loss: 2.3054 - learning_rate: 0.0010
Epoch 3/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.4201 - loss: 1.5245
⚠️ Epoch 3: Val-Accuracy không tăng (Hiện tại: 0.1050 - Kỷ lục: 0.1175)
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.4204 - loss: 1.5235 - val_accuracy: 0.1050 - val_loss: 2.3129 - learning_rate: 0.0010
Epoch 4/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.4577 - loss: 1.3746
⚠️ Epoch 4: Val-Accuracy không tăng (Hiện tại: 0.1050 - Kỷ lục: 0.1175)
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 


🔥 [KỶ LỤC MỚI] Epoch 7: Val-Accuracy tăng thêm 0.0200! (Từ 0.1175 -> 0.1375)
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.5959 - loss: 1.0892 - val_accuracy: 0.1375 - val_loss: 2.2983 - learning_rate: 5.0000e-04
Epoch 8/50
49/50 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.6665 - loss: 0.9797


🔥 [KỶ LỤC MỚI] Epoch 8: Val-Accuracy tăng thêm 0.0175! (Từ 0.1375 -> 0.1550)
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - accuracy: 0.6659 - loss: 0.9800 - val_accuracy: 0.1550 - val_loss: 2.2119 - learning_rate: 5.0000e-04
Epoch 9/50
49/50 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - accuracy: 0.6768 - loss: 0.9356


🔥 [KỶ LỤC MỚI] Epoch 9: Val-Accuracy tăng thêm 0.1675! (Từ 0.1550 -> 0.3225)
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.6768 - loss: 0.9358 - val_accuracy: 0.3225 - val_loss: 1.9625 - learning_rate: 5.0000e-04
Epoch 10/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.6959 - loss: 0.8789
🔥 [KỶ LỤC MỚI] Epoch 10: Val-Accuracy tăng thêm 0.0900! (Từ 0.3225 -> 0.4125)

...


🔥 [KỶ LỤC MỚI] Epoch 29: Val-Accuracy tăng thêm 0.0075! (Từ 0.9550 -> 0.9625)
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - accuracy: 0.9761 - loss: 0.1529 - val_accuracy: 0.9625 - val_loss: 0.1633 - learning_rate: 5.0000e-04
Epoch 30/50
49/50 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.9621 - loss: 0.1576
⚠️ Epoch 30: Val-Accuracy không tăng (Hiện tại: 0.9425 - Kỷ lục: 0.9625)
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - accuracy: 0.9622 - loss: 0.1576 - val_accuracy: 0.9425 - val_loss: 0.1876 - learning_rate: 5.0000e-04
Epoch 31/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.9700 - loss: 0.1257
⚠️ Epoch 31: Val-Accuracy không tăng (Hiện tại: 0.9600 - Kỷ lục: 0.9625)
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - accuracy: 0.9700 - loss: 0.1257 - val_accuracy: 0.9600 - val_loss: 0.1605 - learning_rate: 5.0000e-04
Epoch 32/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.9806 - loss: 0.1211
⚠️ Epoch 32: Val-Accuracy không tăng (Hiện tại: 0.9600 - Kỷ lục: 0.9625)
50/50 ━━━━━━


50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.9806 - loss: 0.0999 - val_accuracy: 0.9675 - val_loss: 0.1603 - learning_rate: 5.0000e-04
Epoch 34/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.9721 - loss: 0.1113
⚠️ Epoch 34: Val-Accuracy không tăng (Hiện tại: 0.9450 - Kỷ lục: 0.9675)
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 46ms/step - accuracy: 0.9721 - loss: 0.1114 - val_accuracy: 0.9450 - val_loss: 0.1901 - learning_rate: 5.0000e-04
Epoch 35/50
49/50 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.9766 - loss: 0.0938
⚠️ Epoch 35: Val-Accuracy không tăng (Hiện tại: 0.9600 - Kỷ lục: 0.9675)
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 0.9768 - loss: 0.0938 - val_accuracy: 0.9600 - val_loss: 0.1314 - learning_rate: 5.0000e-04
Epoch 36/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.9837 - loss: 0.0896
⚠️ Epoch 36: Val-Accuracy không tăng (Hiện tại: 0.9425 - Kỷ lục: 0.9675)
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.9836 - loss: 0.0897 - val_accuracy: 


🔥 [KỶ LỤC MỚI] Epoch 39: Val-Accuracy tăng thêm 0.0100! (Từ 0.9675 -> 0.9775)
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 0.9897 - loss: 0.0549 - val_accuracy: 0.9775 - val_loss: 0.1074 - learning_rate: 5.0000e-04
Epoch 40/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9926 - loss: 0.0517
⚠️ Epoch 40: Val-Accuracy không tăng (Hiện tại: 0.9500 - Kỷ lục: 0.9775)
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 50ms/step - accuracy: 0.9925 - loss: 0.0517 - val_accuracy: 0.9500 - val_loss: 0.1492 - learning_rate: 5.0000e-04
Epoch 41/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.9831 - loss: 0.0644
⚠️ Epoch 41: Val-Accuracy không tăng (Hiện tại: 0.9700 - Kỷ lục: 0.9775)
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 0.9832 - loss: 0.0643 - val_accuracy: 0.9700 - val_loss: 0.1152 - learning_rate: 5.0000e-04
Epoch 42/50
49/50 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.9966 - loss: 0.0468
⚠️ Epoch 42: Val-Accuracy không tăng (Hiện tại: 0.9700 - Kỷ lục: 0.9775)
50/50 ━━━━━━


🔥 [KỶ LỤC MỚI] Epoch 45: Val-Accuracy tăng thêm 0.0025! (Từ 0.9775 -> 0.9800)
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 51ms/step - accuracy: 0.9993 - loss: 0.0324 - val_accuracy: 0.9800 - val_loss: 0.0973 - learning_rate: 2.5000e-04
Epoch 46/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.9998 - loss: 0.0345


🔥 [KỶ LỤC MỚI] Epoch 46: Val-Accuracy tăng thêm 0.0100! (Từ 0.9800 -> 0.9900)
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - accuracy: 0.9998 - loss: 0.0345 - val_accuracy: 0.9900 - val_loss: 0.0713 - learning_rate: 2.5000e-04
Epoch 47/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.9974 - loss: 0.0330
⚠️ Epoch 47: Val-Accuracy không tăng (Hiện tại: 0.9825 - Kỷ lục: 0.9900)
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.9974 - loss: 0.0330 - val_accuracy: 0.9825 - val_loss: 0.0747 - learning_rate: 2.5000e-04
Epoch 48/50
49/50 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.9992 - loss: 0.0290
⚠️ Epoch 48: Val-Accuracy không tăng (Hiện tại: 0.9775 - Kỷ lục: 0.9900)
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 57ms/step - accuracy: 0.9991 - loss: 0.0292 - val_accuracy: 0.9775 - val_loss: 0.0933 - learning_rate: 2.5000e-04
Epoch 49/50
50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9981 - loss: 0.0328
⚠️ Epoch 49: Val-Accuracy không tăng (Hiện tại: 0.9725 - Kỷ lục: 0.9900)
50/50 ━━━━━━

In [9]:
import tensorflow as tf
import os
import numpy as np

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN
# ==========================================
BASE_PATH = "C:/DUT/Ki 2/Edge AI/dataset/"
TRAIN_DIR = os.path.join(BASE_PATH, "train")
H5_MODEL_PATH = "best_model.h5"
TFLITE_OUTPUT_PATH = "best_model.tflite"

# ==========================================
# 1. TẠO REPRESENTATIVE DATASET
# ==========================================
# Bước này cực kỳ quan trọng để đạt Accuracy cao sau khi nén
def representative_data_gen():
    # Nạp một ít dữ liệu từ tập train (khoảng 100-200 ảnh)
    train_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR,
        labels='inferred',
        batch_size=1,
        image_size=(32, 32)
    ).take(150) # Lấy 150 ảnh mẫu

    for input_value, _ in train_ds:
        # Quan trọng: Phải chuẩn hóa giống hệt lúc Train (1./255)
        input_value = input_value / 255.0
        yield [input_value.numpy().astype(np.float32)]

# ==========================================
# 2. TIẾN HÀNH CONVERT & QUANTIZE
# ==========================================
print("🔄 Đang chuyển đổi mô hình sang INT8...")

# Tải mô hình Keras
model = tf.keras.models.load_model(H5_MODEL_PATH)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

# Ép buộc tất cả các phép toán sang Integer (Yêu cầu của ESP32-S3)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8   # Đầu vào ảnh là int8
converter.inference_output_type = tf.int8  # Đầu ra nhãn là int8

tflite_model_quant = converter.convert()

# ==========================================
# 3. LƯU FILE
# ==========================================
with open(TFLITE_OUTPUT_PATH, 'wb') as f:
    f.write(tflite_model_quant)

print(f"✅ Đã lưu file: {TFLITE_OUTPUT_PATH}")
print(f"📏 Kích thước file gốc (.h5): {os.path.getsize(H5_MODEL_PATH)/1024:.2f} KB")
print(f"⚡ Kích thước sau nén (.tflite): {len(tflite_model_quant)/1024:.2f} KB")
print("🚀 Mô hình đã sẵn sàng để nạp lên ESP32-S3!")

🔄 Đang chuyển đổi mô hình sang INT8...
INFO:tensorflow:Assets written to: C:\Users\ngong\AppData\Local\Temp\tmpwjevdjco\assets


INFO:tensorflow:Assets written to: C:\Users\ngong\AppData\Local\Temp\tmpwjevdjco\assets


Saved artifact at 'C:\Users\ngong\AppData\Local\Temp\tmpwjevdjco'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 3), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  1986692094800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1986714852752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1986714859088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1986714861200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1986714860432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1986714861008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1986714859664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1986714857936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1986714857744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1986714861392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1986714

c:\Users\ngong\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\lite\python\convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Found 2000 files belonging to 10 classes.
✅ Đã lưu file: best_model.tflite
📏 Kích thước file gốc (.h5): 433.31 KB
⚡ Kích thước sau nén (.tflite): 49.29 KB
🚀 Mô hình đã sẵn sàng để nạp lên ESP32-S3!
